## Concept Graph ki jagah ConceptNet ka use

> **Note:** Original [Microsoft Concept Graph](https://concept.research.microsoft.com/) API ab available nahi hai. Is notebook ko [ConceptNet](https://conceptnet.io/) use karne ke liye update kar diya gaya hai, jo ek freely available open knowledge graph hai aur jisme concepts ke beech similar `is-a` relationships hote hain.

[ConceptNet](https://conceptnet.io/) concepts ka ek bada semantic network hai, jisme `IsA`, `PartOf`, `UsedFor` aur doosre relationships shamil hain. Yeh available hai:
 * ek downloadable data file ke roop mein
 * ek REST API ke roop mein (koi API key ki zarurat nahi)

ConceptNet ke statistics:
 * 8 million se zyada nodes
 * 83 languages mein 21+ million edges

## ConceptNet web service ka use karna

[ConceptNet](https://conceptnet.io/) concepts ke beech `is-a` (IsA) relationships dhoondhne ke liye ek REST API deta hai. Koi API key ki zarurat nahi hai.  
Call karne ke liye ek sample URL yeh hai: `https://api.conceptnet.io/query?start=/c/en/microsoft&rel=/r/IsA&limit=10`

In [ ]:
import urllib
import json

def http(x):
    response = urllib.request.urlopen(x)
    data = response.read()
    return data.decode('utf-8')

def query(x):
    concept = x.lower().replace(' ', '_')
    url = "https://api.conceptnet.io/query?start=/c/en/{}&rel=/r/IsA&limit=10".format(
        urllib.parse.quote(concept))
    try:
        result = json.loads(http(url))
    except Exception:
        return {}
    edges = result.get('edges', [])
    if not edges:
        return {}
    total_weight = sum(edge['weight'] for edge in edges)
    if total_weight == 0:
        return {}
    return {edge['end']['label']: edge['weight'] / total_weight for edge in edges}

query('microsoft')

Chalo news headlines ko unke parent concepts ke hisaab se classify karne ki koshish karte hain. News headlines lene ke liye hum [NewsApi.org](http://newsapi.org) service use karenge. Service use karne ke liye aapko apni khud ki API key leni hogi - website par jaiye aur free developer plan ke liye register kar lijiye.

In [ ]:
newsapi_key = '<your API key here>'
def get_news(country='us'):
    res = json.loads(http("https://newsapi.org/v2/top-headlines?country={0}&apiKey={1}".format(country,newsapi_key)))
    return res['articles']

all_titles = [x['title'] for x in get_news('us')+get_news('gb')]

In [ ]:
all_titles

Sabse pehle, hum news headlines se nouns nikaalna chahte hain. Iske liye hum `TextBlob` library use karenge, jo is tarah ke kai common NLP tasks ko simple bana deti hai.

In [ ]:
import sys
!{sys.executable} -m pip install textblob
!{sys.executable} -m textblob.download_corpora
from textblob import TextBlob

In [ ]:
w = {}
for x in all_titles:
    for n in TextBlob(x).noun_phrases:
        if n in w:
            w[n].append(x)
        else:
            w[n]=[x]
{ x:len(w[x]) for x in w.keys()}

Hum dekh sakte hain ki nouns humein bade thematic groups nahi dete. Chalo nouns ko Concept Graph se mile zyada general terms se replace karte hain. Isme thoda time lagega, kyunki hum har noun phrase ke liye ek REST call kar rahe hain.

In [ ]:
w = {}
for x in all_titles:
    for noun in TextBlob(x).noun_phrases:
        terms = query(noun)
        for term in [u for u in terms.keys() if terms[u]>0.1]:
            if term in w:
                w[term].append(x)
            else:
                w[term]=[x]

In [ ]:
{ x:len(w[x]) for x in w.keys() if len(w[x])>3}

In [ ]:
print('\nECONOMY:\n'+'\n'.join(w['economy']))
print('\nNATION:\n'+'\n'.join(w['nation']))
print('\nPERSON:\n'+'\n'.join(w['person']))

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->